#Reading From CSV File

In [0]:
df = (
      spark.read.option("header", "true")
      .option("inferSchema", "true")
      .csv("/Volumes/workspace/bronze/source_systems/source_crm/sales_details.csv")
      )
df.display()

#Write it to Bronze Layer

In [0]:
df.write.mode("overwrite").saveAsTable("workspace.bronze.crm_sales_details")

In [0]:
%sql
Select * FROM workspace.bronze.crm_sales_details

#Define Ingestion Layer (Config)

### Prevent repeating ingestion steps

In [0]:
INGESTION_CONFIG = [
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/source_systems/source_crm/cust_info.csv",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/source_systems/source_crm/prd_info.csv",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Volumes/workspace/bronze/source_systems/source_crm/sales_details.csv",
        "table": "crm_sales_details"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/source_systems/source_erp/CUST_AZ12.csv",
        "table": "erp_cust_az12"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/source_systems/source_erp/LOC_A101.csv",
        "table": "erp_loc_a101"
    },
    {
        "source": "erp",
        "path": "/Volumes/workspace/bronze/source_systems/source_erp/PX_CAT_G1V2.csv",
        "table": "erp_px_cat_g1v2"
    }
]


#Ingest files into the Bronze Layer

In [0]:
%python

for item in INGESTION_CONFIG:
    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")

    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(item["path"])
    )

    (
        df.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable(f"workspace.bronze.{item['table']}")
    )